#Импорт библиотек и метрик


In [ ]:
pip install CatBoost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from metric import precision_at_recall

#Загрузка данных


In [5]:
train = pd.read_csv('/content/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('/content/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('/content/events.csv', parse_dates=['event_ts'])

#Провожу фильтрацию в окне наблюдений


In [6]:
def events_in_window(ev, meta):
    ev = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

#Сортрую события по кукам и времени

In [17]:
def extract_powerful_features(ev, meta):
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()

    ev['ts_diff'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

    ua = ev['user_agent'].fillna('').str.lower()
    ev['is_headless'] = ua.str.contains('headless|selenium|phantom|puppeteer', regex=True).astype(int)
    ev['is_mobile'] = ua.str.contains('mobile|android|iphone|ipad', regex=True).astype(int)

    #Сворачиваю поток событий в одну строку на куку - по одному агрегату на признак
    aggs = ev.groupby('cookie_id').agg(
    n_events=('event_ts', 'count'),
    duration=('event_ts', lambda s: (s.max() - s.min()).total_seconds()),

    # Разнообразие
    item_nunique=('item_id', 'nunique'),
    cat_nunique=('item_category', 'nunique'),
    loc_nunique=('item_location', 'nunique'),
    query_nunique=('search_query', 'nunique'),

    # Временная статистика
    ts_diff_min=('ts_diff', 'min'),
    ts_diff_median=('ts_diff', 'median'),
    ts_diff_mean=('ts_diff', 'mean'),
    ts_diff_std=('ts_diff', 'std'),
    ts_diff_max=('ts_diff', 'max'),

    # Глубина поиска
    page_max=('search_page', 'max'),
    page_mean=('search_page', 'mean'),

    # Поведение курсора (СУПЕР-ФИЧИ)
    pointer_count=('pointer_x', 'count'),
    pointer_x_std=('pointer_x', 'std'),
    pointer_y_std=('pointer_y', 'std'),

    # Маркеры бота
    headless_count=('is_headless', 'sum'),
    mobile_ratio=('is_mobile', 'mean')
    )

    #Строию матрицу "кука x тип события" с счётчиками
    event_types = ev.pivot_table(index='cookie_id', columns='event_name', aggfunc='size', fill_value=0)

    event_types_ratio = event_types.div(event_types.sum(axis=1), axis=0).add_prefix('ratio_ev_')


    #Склеиваю агрегаты и доли событий по индексу 'cookie_id'
    features = aggs.join(event_types_ratio).reset_index()


    meta_df = meta[['cookie_id', 'cookie_created_at', 'window_start_ts']].copy()
    meta_df['cookie_age_days'] = (meta_df['window_start_ts'] - meta_df['cookie_created_at']).dt.total_seconds() / 86400.0

    res = meta_df[['cookie_id', 'cookie_age_days']].merge(features, on='cookie_id', how='left')

    #Считаею четыре производных отношения, затем одним вызовом закрываею все оставшиеся пропуски во всей таблице разом
    res['events_per_second'] = res['n_events'] / (res['duration'] + 1.0)
    res['item_per_event'] = res['item_nunique'] / (res['n_events'] + 1.0)
    res['cat_per_item'] = res['cat_nunique'] / (res['item_nunique'] + 1.0)
    res['pointer_ratio'] = res['pointer_count'] / (res['n_events'] + 1.0)

    return res.fillna(0)







#Строю признаковые таблицы отдельно для train и test — на их собственных, уже отфильтрованных по окну событиях

In [18]:
X_train = extract_powerful_features(ev_tr, train)
X_test = extract_powerful_features(ev_te, test)
y_train = train['target'].values

#Валидация: сплит по времени и обучение с early stopping

In [19]:
is_valid = train.window_start_ts.ge('2026-04-17').values
feature_cols = [c for c in X_train.columns if c != 'cookie_id']

model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    eval_metric='Logloss',
    random_seed=42,
    thread_count=-1,
    verbose=100
)

model.fit(
    X_train.loc[~is_valid, feature_cols], y_train[~is_valid],
    eval_set=(X_train.loc[is_valid, feature_cols], y_train[is_valid]),
    early_stopping_rounds=150
)

p_va = model.predict_proba(X_train.loc[is_valid, feature_cols])[:, 1]
val_score = precision_at_recall(y_train[is_valid], p_va)
print(f'\n>>> P@R0.7 на валидации: {val_score:.4f} <<<\n')

0:	learn: 0.6494975	test: 0.6503773	best: 0.6503773 (0)	total: 74.7ms	remaining: 1m 51s
100:	learn: 0.1361304	test: 0.1580552	best: 0.1580552 (100)	total: 1.13s	remaining: 15.6s
200:	learn: 0.1113985	test: 0.1428068	best: 0.1428068 (200)	total: 2.19s	remaining: 14.1s
300:	learn: 0.0977391	test: 0.1385631	best: 0.1385622 (299)	total: 3.27s	remaining: 13s
400:	learn: 0.0874362	test: 0.1360895	best: 0.1360895 (400)	total: 4.33s	remaining: 11.9s
500:	learn: 0.0800361	test: 0.1344688	best: 0.1344688 (500)	total: 6.02s	remaining: 12s
600:	learn: 0.0733627	test: 0.1341595	best: 0.1341048 (594)	total: 7.75s	remaining: 11.6s
700:	learn: 0.0672698	test: 0.1344888	best: 0.1340953 (601)	total: 8.96s	remaining: 10.2s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.1340953396
bestIteration = 601

Shrink model to first 602 iterations.

>>> P@R0.7 на валидации: 0.7619 <<<



#Финальное обучение и сохранение предсказания

In [21]:
best_iters = model.get_best_iteration() + 1
print(f"Обучение финальной модели на всем датасете ({best_iters} итераций)...")

final_model = CatBoostClassifier(
    iterations=best_iters,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    thread_count=-1,
    verbose=100
)

final_model.fit(X_train[feature_cols], y_train)

sub = pd.DataFrame({
    'cookie_id': X_test['cookie_id'],
    'score': final_model.predict_proba(X_test[feature_cols])[:, 1]
})

sub.to_csv('submission.csv', index=False)

Обучение финальной модели на всем датасете (602 итераций)...
0:	learn: 0.6500706	total: 11.9ms	remaining: 7.17s
100:	learn: 0.1370760	total: 1.22s	remaining: 6.05s
200:	learn: 0.1149988	total: 2.98s	remaining: 5.95s
300:	learn: 0.1028121	total: 5.02s	remaining: 5.02s
400:	learn: 0.0928086	total: 6.29s	remaining: 3.15s
500:	learn: 0.0850683	total: 7.46s	remaining: 1.5s
600:	learn: 0.0797157	total: 8.68s	remaining: 14.4ms
601:	learn: 0.0795783	total: 8.69s	remaining: 0us
